# 我的对抗性对话
J. 麦金纳尼，2025 年 5 月 26 日
我正在从第 2 周第 1 天笔记本中取出一些单元格并对其进行修改，以便我可以在 OpenAI 和本地法学硕士 (gemma3:12b) 之间进行敌对对话。  首先，我将重新实现 Ed 在第 2 周第 1 天笔记本中所做的事情。  然后我会尝试进行更深入的对话。

In [ ]:
# 进口

import os
from dotenv import load_dotenv
from openai import OpenAI
#进口人择
from IPython.display import Markdown, display, update_display

In [ ]:
# 在名为 .env 的文件中加载环境变量
# 打印键前缀以帮助进行任何调试

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    


In [ ]:
# 连接到 OpenAI、Anthropic
openai = OpenAI()

## 聊天机器人之间的对抗性对话..

您已经熟悉将提示组织成列表，例如：

````
[
    {"role": "系统", "content": "这里是系统消息"},
    {"role": "user", "content": "此处提示用户"}
]
````

事实上，这个结构可以用来反映更长的对话历史：

````
[
    {"role": "系统", "content": "这里是系统消息"},
    {"role": "user", "content": "此处提示第一个用户"},
    {"role": "助理", "content": "助理的回应"},
    {"role": "user", "content": "新用户提示"},
]
````

我们可以利用这种方式与历史进行更长时间的互动。

In [ ]:
# 让我们在 GPT-4o-mini 和 Gemma3:12b 之间进行对话
# 我们使用廉价版本的模型，因此成本将是最低的

gpt_model = "gpt-4o-mini"
local_model = 'gemma3:12b'

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

local_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
local_messages = ["Hi"]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, local in zip(gpt_messages, local_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": local})
    completion = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    return completion.choices[0].message.content

In [ ]:
call_gpt()

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
basellm = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
def call_local():
    messages = []
    for gpt, local_message in zip(gpt_messages, local_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": local_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    
    completion = basellm.chat.completions.create(
        model=local_model,
        messages=messages
    )
    
    return completion.choices[0].message.content

In [ ]:
call_local()

In [ ]:
call_gpt()

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
gpt_messages = ["Hi there"]
local_messages = ["Hi"]

print(f"GPT:\n{gpt_messages[0]}\n")
print(f"local:\n{local_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    local_next = call_local()
    print(f"local:\n{local_next}\n")
    local_messages.append(local_next)

## 让我们尝试进行更深思熟虑的对话
这两个聊天机器人将就美国是否应该在 1917 年参加第一次世界大战进行友好讨论。它们都很开放，可以互相学习。

In [ ]:
# 让我们在 GPT-4o-mini 和 Gemma3:12b 之间进行对话
# 我们使用廉价版本的模型，因此成本将是最低的

gpt_system = "You are a chatbot who believes it was a mistake for the US to enter World War I; \
you are open to other arguments, but you feel the evidence suggests the world would have been \
better off if the US had stayed isolationalist.   You consider counter arguments but also express \
your own arguments."

local_system = "You are a chatbot who believes the US made the right decision entering World War I in \
1917.  Overall, the world is a better place for it.  You are open minded but believe the evidence \
supports this view.  You consider counter arguments but also express your own arguments."

gpt_messages = ["It was such a mistake for the US to enter WWI"]
local_messages = ["Why do you say that?"]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
print(f"GPT:\n{gpt_messages[0]}\n")
print(f"local:\n{local_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    local_next = call_local()
    print(f"local:\n{local_next}\n")
    local_messages.append(local_next)

## 结论
我对这次谈话的富有洞察力感到惊讶。  他们不仅探索了所有的利弊，还开始将这些经验教训应用到当今的外交政策中。  这看起来是探索主题的一个非常好的方法。